# 03 — Score Trends

Canonical ensemble trajectories, season-week growth, and first-pass feature engineering. Run `uv run python scripts/derive.py --rebuild` before this notebook so canonical/week fields are current.


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('../../scores.db')
df = pd.read_sql('SELECT * FROM v_performances_canonical', conn)
df['performance_date'] = pd.to_datetime(df['performance_date'])
df = df.sort_values(['canonical_ensemble_id', 'season_year', 'performance_date'])
df.head()


In [ ]:
# Median total score by season and class
yearly = df.groupby(['season_year', 'class_code'])['total_score'].median().unstack()
yearly.plot(figsize=(12, 5), marker='o', title='Median total score by season (per class)')
plt.tight_layout()


In [ ]:
# Within-season score growth using SCPA season week
weekly = (
    df.groupby(['season_year', 'season_week_calendar'])['total_score']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for yr, grp in weekly.groupby('season_year'):
    ax.plot(grp['season_week_calendar'], grp['total_score'], marker='o', label=yr, alpha=0.7)
ax.set_xlabel('SCPA season week')
ax.set_ylabel('Mean total score')
ax.set_title('Within-season score growth by SCPA season week')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()


In [ ]:
# Top canonical ensembles by number of seasons competed
seasons_per_ensemble = (
    df.groupby(['canonical_ensemble_id', 'canonical_ensemble_name'])['season_year']
    .nunique()
    .sort_values(ascending=False)
)
top_ensembles = seasons_per_ensemble.head(10).index.get_level_values('canonical_ensemble_id').tolist()
seasons_per_ensemble.head(20)


In [ ]:
# Peak annual score trajectory for top canonical ensembles
peak = (
    df[df['canonical_ensemble_id'].isin(top_ensembles)]
    .groupby(['canonical_ensemble_id', 'canonical_ensemble_name', 'season_year'])['total_score']
    .max()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
for (_, name), grp in peak.groupby(['canonical_ensemble_id', 'canonical_ensemble_name']):
    ax.plot(grp['season_year'], grp['total_score'], marker='o', label=name)
ax.set_title('Peak annual score — top canonical ensembles by seasons competed')
ax.set_xlabel('Season')
ax.set_ylabel('Peak total score')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()


In [ ]:
# First-pass feature table for modeling
features = (
    df.groupby(['canonical_ensemble_id', 'canonical_ensemble_name', 'season_year'])
    .agg(
        show_count=('performance_key', 'count'),
        debut_week=('season_week_calendar', 'min'),
        last_week=('season_week_calendar', 'max'),
        peak_score=('total_score', 'max'),
        final_score=('total_score', lambda s: s.iloc[-1]),
        best_placement=('placement', 'min'),
        classes=('class_code', lambda s: ','.join(sorted(set(s)))),
        made_finals=('is_finals', 'max'),
        competed_championship=('is_championship', 'max'),
    )
    .reset_index()
    .sort_values(['canonical_ensemble_id', 'season_year'])
)
features['prior_peak_score'] = features.groupby('canonical_ensemble_id')['peak_score'].shift(1)
features['prior_made_finals'] = features.groupby('canonical_ensemble_id')['made_finals'].shift(1).fillna(0).astype(int)
features['score_growth_vs_prior_peak'] = features['peak_score'] - features['prior_peak_score']
features.head(30)


In [ ]:
# Debut timing vs peak score, by class mix
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=features, x='debut_week', y='peak_score', hue='made_finals', ax=ax)
ax.set_title('Debut week vs peak score')
ax.set_xlabel('Debut SCPA season week')
ax.set_ylabel('Peak total score')
plt.tight_layout()
